# 💬 Análisis de Sentimientos en Reviews (NLP & Classification)

> Clasificación supervisada de reseñas de clientes en portugués utilizando **TF-IDF Vectorizer**, **Multinomial Naive Bayes** y **Linear Support Vector Classifier (LinearSVC)** con balanceo de clases y validación temporal estricta.

---

## 🎯 Objetivo de Negocio

Detectar de forma automatizada y temprana el sentimiento de los clientes (Positivo vs. Negativo) a partir del texto de sus reseñas para priorizar la atención al cliente, identificar fallas operativas y proteger el NPS de la plataforma.

```mermaid
flowchart LR
    A[olist_order_reviews_dataset.csv] --> B[src.common.data]
    B --> C[src.sentiment.features]
    C --> D[TF-IDF + LinearSVC / MultinomialNB]
    D --> E[src.sentiment.viz]
```


In [ ]:
# Librerías estándar y de terceros
import pandas as pd
from sklearn.model_selection import train_test_split

# Módulos del proyecto (instalados en modo editable via pyproject.toml)
from src.common.data import load_sentiment_raw_data
from src.sentiment.features import (
    build_sentiment_features,
    build_sentiment_pipelines,
    evaluate_sentiment_models,
    get_portuguese_stopwords,
)
from src.sentiment.viz import plot_confusion_matrices, plot_top_coefficients


## 1. Carga del Dataset de Reseñas

Cargamos las reseñas originales de Olist mediante el cargador centralizado de `src.common.data`.

In [ ]:
df_reviews = load_sentiment_raw_data()
print(f"Total de registros originales: {len(df_reviews):,}")
df_reviews.head(3)


## 2. Preprocesamiento Textual e Ingeniería de Features

Construcción del dataset procesado mediante `build_sentiment_features`:
- Filtra registros sin contenido textual (descarta reseñas sin título ni mensaje).
- Elimina valoraciones neutras (3 estrellas) y define la variable target binaria (Positivo $\ge 4$, Negativo $\le 2$).
- Ordena cronológicamente según `review_creation_date`.
- Combina título y mensaje normalizando espacios, puntuación y mayúsculas.

In [ ]:
df_sorted = build_sentiment_features(df_reviews)

nuevas = (
    df_sorted["review_comment_message"].isna() & df_sorted["review_comment_title"].notna()
).sum()
print(f"Muestras finales ordenadas por fecha: {len(df_sorted):,}")
print(f"Registros aportados por el título (sin mensaje): {nuevas:,}")

min_date = df_sorted["review_creation_date"].min().strftime("%Y-%m-%d")
max_date = df_sorted["review_creation_date"].max().strftime("%Y-%m-%d")
print(f"Rango de fechas: desde {min_date} hasta {max_date}")

df_sorted[["review_score", "target", "review_text_full", "review_creation_date"]].head(3)


## 3. División Temporal (Train/Test Split)

Aplicamos un particionado cronológico estricto (80% pasado para entrenamiento, 20% futuro más reciente para evaluación) con `shuffle=False` para prevenir *Data Leakage*.

In [ ]:
X = df_sorted["review_text_full"]
y = df_sorted["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Muestras de entrenamiento (pasado): {len(X_train):,}")
print(f"Muestras de evaluación (futuro reciente): {len(X_test):,}")

print("\nDistribución de clases en Entrenamiento:")
print(y_train.value_counts(normalize=True).rename({1: "Positivo (1)", 0: "Negativo (0)"}))

print("\nDistribución de clases en Evaluación (Test):")
print(y_test.value_counts(normalize=True).rename({1: "Positivo (1)", 0: "Negativo (0)"}))


## 4. Entrenamiento de Pipelines (Naive Bayes vs Linear SVM)

Configuración de stopwords personalizadas en portugués (conservando palabras de negación como `nao`, `nunca`, `jamais`) y evaluación de los pipelines `MultinomialNB` vs `LinearSVC(class_weight="balanced")`.

In [ ]:
stop_words = get_portuguese_stopwords(exclude_negations=True)
pipelines = build_sentiment_pipelines(stop_words=stop_words)

df_comparison = evaluate_sentiment_models(pipelines, X_train, y_train, X_test, y_test)
df_comparison


## 5. Evaluación Comparativa y Matrices de Confusión

Visualización de las matrices de confusión para contrastar el rendimiento entre ambos modelos, prestando especial atención al *Recall* sobre la clase Negativa.

In [ ]:
plot_confusion_matrices(pipelines, X_test, y_test)


## 6. Importancia de Características (Top 15 Palabras en Linear SVM)

Extracción de los coeficientes del modelo `LinearSVC` para interpretar qué términos aportan mayor peso hacia las polaridades Negativa y Positiva.

In [ ]:
neg, pos = plot_top_coefficients(pipelines["Linear SVM (Balanced)"], top_n=15)


## 7. Conclusiones e Impacto en el Negocio

1. **Superioridad del Linear SVM**: Gracias al parámetro `class_weight="balanced"`, el modelo Linear SVM alcanza un **Recall de 93.26% en la clase negativa**, superando el 89.65% de Naive Bayes.
2. **Impacto Operativo**: En atención al cliente y soporte e-commerce, minimizar los falsos negativos (reseñas negativas no detectadas) es crucial para activar alertas operativas y mitigar la pérdida de clientes (*churn*).
3. **Interpretabilidad**: Los términos con mayor peso negativo reflejan fallas logísticas y de producto (`nao recomendo`, `pessimo`, `nao recebi`, `defeito`), mientras que los positivos destacan cumplimiento y satisfacción (`otimo`, `excelente`, `recomendo`, `parabens`).
